# 🎯 Ultimate Tic-Tac-Toe — AlphaZero Trainer

Train the game's AI **entirely in your browser** — no terminal required.

This notebook runs an AlphaZero-style self-play loop (MCTS + a small ConvNet).
It saves checkpoints to **your Google Drive**, so if Colab disconnects you just
re-run the cells and training **resumes where it left off**.

**The plan**
1. Turn on a free GPU
2. Connect Google Drive (for saving)
3. Pull the code
4. Install dependencies
5. Train (run as long as you like; resumable)
6. Export the model and put it in your app

> Tip: training gets stronger the longer you run it. A few hours on a free GPU
> already produces a respectable opponent.

## Step 1 — Turn on a GPU

In the Colab menu: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.

Then run the cell below to confirm.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — training will still work but be slower. "
          "Go to Runtime → Change runtime type → T4 GPU.")

## Step 2 — Connect Google Drive

This is where checkpoints and the exported model are saved, so your progress
survives disconnects. Click through the authorization popup when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/uttt-alphazero'   # change if you like
CKPT_DIR  = f'{DRIVE_DIR}/checkpoints'
ONNX_OUT  = f'{DRIVE_DIR}/uttt.onnx'
os.makedirs(CKPT_DIR, exist_ok=True)
print("Saving checkpoints to:", CKPT_DIR)
print("Exporting model to:   ", ONNX_OUT)

## Step 3 — Get the code

Pulls this repository into Colab. If your repo is **private**, paste a GitHub
token (Settings → Developer settings → Personal access tokens) into
`GITHUB_TOKEN`. For a public repo, leave it blank.

Set `BRANCH` to the branch that has the code. If you haven't merged to `main`
yet, use your feature branch (e.g. `claude/nifty-cannon-2rxz80`).

In [ ]:
REPO   = "nwlb91/ultimatettt"   # owner/name
BRANCH = "main"                  # branch that has the code
GITHUB_TOKEN = ""                # only needed for a PRIVATE repo

import os
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
url  = f"https://{auth}github.com/{REPO}.git"
DEST = "/content/ultimatettt"

if os.path.exists(DEST):
    !cd {DEST} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {url} {DEST}

%cd {DEST}
import sys
if DEST not in sys.path:
    sys.path.insert(0, DEST)

# Sanity check: did we actually get the code on this branch?
if not os.path.exists("training/requirements.txt"):
    raise SystemExit(
        f"\n>>> The code was not found on branch '{BRANCH}'.\n"
        f">>> Set BRANCH above to the branch that has the app "
        f"(e.g. 'claude/nifty-cannon-2rxz80'), then re-run this cell.\n")
print("Working dir:", os.getcwd(), "(code found OK)")

## Step 4 — Install dependencies

Torch is already on Colab, so this is quick.

In [ ]:
!pip install -q -r training/requirements.txt && echo "Dependencies ready (OK)" || echo ">>> Install failed -- re-check Step 3 (is BRANCH correct?)"

## Step 5 — Sanity check

Quickly confirm the game engine and network import correctly.

In [ ]:
from uttt.game import Game
from training.model import UTTTNet
g = Game()
print("Legal opening moves:", len(g.legal_moves()))     # should be 81
net = UTTTNet(channels=64, blocks=5)
params = sum(p.numel() for p in net.parameters())
print(f"Network parameters: {params:,}")

## Step 6 — Train 🚀

This is the main loop. It **resumes automatically** from `CKPT_DIR/latest.pt`,
so you can stop and re-run this cell any time (e.g. after a Colab disconnect).

**Knobs (trade speed for strength):**
- `iterations` — how many self-play→train cycles to run *this session*.
- `games_per_iter` — self-play games per cycle (more = better data, slower).
- `sims` — MCTS simulations per move (more = stronger play & better targets, slower).
- `channels` / `blocks` — network size. Bigger = stronger & slower.

Defaults below are a solid balance for a free T4. Start small, then increase
`iterations` and re-run to keep improving. The model is exported to Drive after
every iteration, so you always have the latest to deploy.

In [ ]:
from training.train import Config, train

cfg = Config(
    # network
    channels=64, blocks=5,
    # search
    sims=100,
    # self-play / data
    games_per_iter=40,
    # optimization
    train_steps=400, batch_size=256, lr=1e-3,
    # loop
    iterations=40,          # raise this and re-run to train longer
    ckpt_dir=CKPT_DIR,      # on Drive -> survives disconnects
    onnx_out=ONNX_OUT,      # on Drive
    export_every=1, save_every=5,
)

net = train(cfg)           # resumes if a checkpoint already exists

## Step 7 — Is it getting stronger? (optional)

Pit the current network (with a little search) against plain Monte-Carlo
rollouts. A win rate above ~0.5 means the net has learned something; it should
climb as you train more.

In [ ]:
import torch
from training.model import load_net
from training.torch_evaluator import TorchEvaluator
from training.arena import arena
from uttt.ai import RolloutEvaluator

device = 'cuda' if torch.cuda.is_available() else 'cpu'
net, _ = load_net(f'{CKPT_DIR}/latest.pt', device)

result = arena(TorchEvaluator(net, device), RolloutEvaluator(), games=20, sims=50)
print(result)
print(f"Network win rate vs rollouts: {result['win_rate']:.0%}")

## Step 8 — Put the model in your app

Your trained network lives at `ONNX_OUT` on Drive. Get it into the app's
`models/uttt.onnx` and Render will redeploy automatically.

**Option A — download, then upload via GitHub (no terminal):**
1. Run the cell below to download `uttt.onnx` to your computer.
2. In your GitHub repo, open the **`models/`** folder → **Add file → Upload files**.
3. Drag in `uttt.onnx` (it must be named exactly `uttt.onnx`), commit to your branch.
4. Render redeploys; the AI now uses your network. (Check `/healthz` → `"model": true`.)

In [ ]:
from google.colab import files
import shutil
shutil.copy(ONNX_OUT, 'uttt.onnx')
files.download('uttt.onnx')

**Option B — push straight from Colab** (needs a token with `repo` scope).
Uncomment and fill in:

In [ ]:
# GITHUB_TOKEN = "ghp_xxx"     # token with repo scope
# EMAIL = "you@example.com"
# NAME  = "Your Name"
# !cp {ONNX_OUT} {DEST}/models/uttt.onnx
# !cd {DEST} && git config user.email "{EMAIL}" && git config user.name "{NAME}" \
#   && git add models/uttt.onnx \
#   && git commit -m "Update trained model" \
#   && git push https://{GITHUB_TOKEN}@github.com/{REPO}.git HEAD:{BRANCH}
print("(Option B is commented out — fill it in if you want to push from Colab.)")

---
### Keep training later
Just reopen this notebook and run Steps 1–6 again. Because checkpoints are on
Drive, training **continues from where it stopped**. Re-export (Step 8) whenever
you want to deploy a stronger model.